# DS/CMPSC 410 Spring 2025
# Instructor: Professor John Yen
# TA: Jin Peng and Jingxi Zhu
# Lab 7: Post-Modeling Analysis: ALS Recommendation Systems
## The goals of this lab are for you to be able to
### - Conduct post-modeling analysis using PySpark to identify weakness of the model.
### - Data used in this lab includes those used in Lab 6 and the **csv file of predicted rating and actual rating for testing data you generated in Lab 6**
### - For ALS-recommendation systems, be able to calculate average errors for each movie and each user.
### - Be able to investigate the relationship between average error and total number of reviews for movies.
### - Be able to investigate the relationship between average error and total number of reviews for users.
### - Be able to identify opportunities to improve the performance of recommendation system based on the result of analysis above.
### - Be able to filter movies and/or users based on the opportunities identified to improve the recommendation system.
### - Be able to perform hyper-parameter tuning on the filtered data and compare its performance with that of the original model.
### - Be able to debug (in local mode) using Restart Kernel if needed.
### - Note: This lab only requires running Spark in the local mode (using small review dataset). 

## Exercises: 
- Exercise 1: 5 points
- Exercise 2: 10 points
- Exercise 3: 15 points
- Exercise 4: 15 points
- Exercise 5: 15 points
- Exercise 6: 15 points
- Exercise 7: 15 points
- Exercise 8: 10 points
## Total Points: 100 points

# Due: midnight, March 2, 2025


## The first thing we need to do in each Jupyter Notebook running pyspark is to import pyspark first.

In [1]:
import pyspark

### Once we import pyspark, we need to import "SparkContext".  Every spark program needs a SparkContext object
### In order to use Spark SQL on DataFrames, we also need to import SparkSession from PySpark.SQL
### In addition, we import ``ALS`` from ``MLlib.recommendation`` of ``pyspark.

In [2]:
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.types import StructField, StructType, StringType, LongType, IntegerType, FloatType
from pyspark.sql.functions import col, column
from pyspark.sql.functions import expr
from pyspark.sql.functions import split
from pyspark.sql import Row
from pyspark.mllib.recommendation import ALS

## We then create a Spark Session variable (rather than Spark Context) in order to use DataFrame. 
- Note: We temporarily use "local" as the parameter for master in this notebook so that we can complete and debug the code using Jupyter Server.  After we export it to .py file for execution in the cluster, however, we need to REMOVE .master("local") in the .py file so that it runs in cluster (Standalone) mode in ICDS cluster when we execute ``pbs-spark-submit``

In [3]:
ss=SparkSession.builder.master("local").appName("Lab7 Post-Model Analysis of ALS Recommendation Systems").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/02/26 14:01:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
ss.sparkContext.setLogLevel("WARN")

In [5]:
ss.sparkContext.setCheckpointDir("/storage/home/ajv5723/scratch")

## Exercise 1 (5 points) (a) Add your name below AND (b) replace the path below with the path of your Lab5 and Lab6 directory.
## Answer for Exercise 1
- a: Your Name: Aidan Vesci

In [6]:
rating_schema = StructType([ StructField("UserID", IntegerType(), False ), \
                            StructField("MovieID", IntegerType(), True), \
                            StructField("Rating", FloatType(), True ), \
                            StructField("RatingID", IntegerType(), True ), \
                           ])

In [7]:
Datapath = '/storage/home/ajv5723/work/Lab5'
Modelpath = '/storage/home/ajv5723/work/Lab6'

In [8]:
ratings_DF = ss.read.csv(f"{Datapath}/ratings_samples.csv", schema=rating_schema, header=True, inferSchema=False)

In [9]:
ratings_DF.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: float (nullable = true)
 |-- RatingID: integer (nullable = true)



In [10]:
ratings2_DF = ratings_DF.select("UserID","MovieID","Rating")

In [11]:
ratings2_DF.first()

Row(UserID=1, MovieID=31, Rating=2.5)

# Read Predicted and Actual Rating of Testing Data Generated from your Jupyter Notebook of Lab6

# Exercise 2 (10 points)
Complete the code below to read the csv file of predicted rating and actual rating of testing data you saved in Lab6.

In [12]:
schema4= StructType([ StructField("UserID", IntegerType(), True), \
                      StructField("MovieID", IntegerType(), True ), \
                      StructField("Actual Rating", FloatType(), True), \
                      StructField("Predicted Rating", FloatType(), True), \
                    ])

In [13]:
testingD_acturalR_predictedR_DF = ss.read.csv(f"{Modelpath}/Lab6ALS_testingD_actualR_predictedR_local.csv", schema=schema4, header=True, inferSchema=False)

In [14]:
testingD_acturalR_predictedR_DF.show(3)

+------+-------+-------------+----------------+
|UserID|MovieID|Actual Rating|Predicted Rating|
+------+-------+-------------+----------------+
|     1|   1029|          3.0|       2.4575818|
|     1|   1061|          3.0|       2.0036154|
|     1|   1293|          2.0|       2.6530988|
+------+-------+-------------+----------------+
only showing top 3 rows



In [15]:
testingD_acturalR_predictedR_DF.count()

19050

# Calculate the Error of Prediction for Testing Data, save the result as a new column in the DataFrame

In [16]:
Error_Testing_DF = testingD_acturalR_predictedR_DF.withColumn("Error", col("Predicted Rating") - col("Actual Rating"))

In [17]:
Error_Testing_DF.show(5)

+------+-------+-------------+----------------+-----------+
|UserID|MovieID|Actual Rating|Predicted Rating|      Error|
+------+-------+-------------+----------------+-----------+
|     1|   1029|          3.0|       2.4575818|-0.54241824|
|     1|   1061|          3.0|       2.0036154| -0.9963846|
|     1|   1293|          2.0|       2.6530988|  0.6530988|
|     1|   1339|          3.5|       1.8667258| -1.6332742|
|     1|   1405|          1.0|        2.023204|  1.0232041|
+------+-------+-------------+----------------+-----------+
only showing top 5 rows



In [18]:
from pyspark.sql.functions import abs

In [19]:
Abs_Err_DF = Error_Testing_DF.withColumn("AbsoluteErr", abs(col("Error")))

In [20]:
Abs_Err_DF.show(5)

+------+-------+-------------+----------------+-----------+-----------+
|UserID|MovieID|Actual Rating|Predicted Rating|      Error|AbsoluteErr|
+------+-------+-------------+----------------+-----------+-----------+
|     1|   1029|          3.0|       2.4575818|-0.54241824| 0.54241824|
|     1|   1061|          3.0|       2.0036154| -0.9963846|  0.9963846|
|     1|   1293|          2.0|       2.6530988|  0.6530988|  0.6530988|
|     1|   1339|          3.5|       1.8667258| -1.6332742|  1.6332742|
|     1|   1405|          1.0|        2.023204|  1.0232041|  1.0232041|
+------+-------+-------------+----------------+-----------+-----------+
only showing top 5 rows



# Calculate, for each User, the total number of reviews, and the average absolute prediction error

In [21]:
User_Err_Sum = Abs_Err_DF.groupBy("UserID").sum("AbsoluteErr")

In [22]:
User_Err_Sum.show(3)

+------+------------------+
|UserID|  sum(AbsoluteErr)|
+------+------------------+
|   148| 7.780052661895752|
|   463|55.948812782764435|
|   471|18.781705856323242|
+------+------------------+
only showing top 3 rows



In [23]:
User_review_total= Abs_Err_DF.groupBy("UserID").count()

In [24]:
User_review_total.show(3)

+------+-----+
|UserID|count|
+------+-----+
|   148|   16|
|   463|   83|
|   471|   41|
+------+-----+
only showing top 3 rows



In [25]:
Joined_User_Error_DF = User_Err_Sum.join(User_review_total, "UserID", "inner")

In [26]:
Joined_User_Error_DF.show(3)

+------+------------------+-----+
|UserID|  sum(AbsoluteErr)|count|
+------+------------------+-----+
|   148| 7.780052661895752|   16|
|   463|55.948812782764435|   83|
|   471|18.781705856323242|   41|
+------+------------------+-----+
only showing top 3 rows



In [27]:
Avg_User_Error_DF = Joined_User_Error_DF.withColumn("AvgErr", col("sum(AbsoluteErr)")/col("count"))

In [28]:
Avg_User_Error_DF.show(3)

+------+------------------+-----+------------------+
|UserID|  sum(AbsoluteErr)|count|            AvgErr|
+------+------------------+-----+------------------+
|   148| 7.780052661895752|   16|0.4862532913684845|
|   463|55.948812782764435|   83|0.6740820817200535|
|   471|18.781705856323242|   41|0.4580903867395913|
+------+------------------+-----+------------------+
only showing top 3 rows



In [29]:
sorted_Avg_User_Error_DF = Avg_User_Error_DF.orderBy("AvgErr", ascending=False)

In [30]:
sorted_Avg_User_Error_DF.show(10)

+------+------------------+-----+------------------+
|UserID|  sum(AbsoluteErr)|count|            AvgErr|
+------+------------------+-----+------------------+
|   477| 6.030291795730591|    3|2.0100972652435303|
|   227| 9.614219665527344|    5|1.9228439331054688|
|   638|1.8666584491729736|    1|1.8666584491729736|
|   375|3.6735706329345703|    2|1.8367853164672852|
|   540| 5.289602279663086|    3|1.7632007598876953|
|    53|10.575591087341309|    6|1.7625985145568848|
|   348|29.448211550712585|   17| 1.732247738277211|
|   364|15.104052782058716|    9| 1.678228086895413|
|   336| 8.343994379043579|    5|1.6687988758087158|
|   504| 9.498573541641235|    6|1.5830955902735393|
+------+------------------+-----+------------------+
only showing top 10 rows



In [31]:
Avg_User_Error_DF.orderBy("count", ascending=True).show(10)

+------+------------------+-----+------------------+
|UserID|  sum(AbsoluteErr)|count|            AvgErr|
+------+------------------+-----+------------------+
|   638|1.8666584491729736|    1|1.8666584491729736|
|   604| 1.205296277999878|    1| 1.205296277999878|
|    64|0.8108320236206055|    1|0.8108320236206055|
|    65|1.1372318267822266|    2|0.5686159133911133|
|   375|3.6735706329345703|    2|1.8367853164672852|
|   556|1.8923592567443848|    2|0.9461796283721924|
|   445|1.1389100551605225|    2|0.5694550275802612|
|   651|1.7106232643127441|    2|0.8553116321563721|
|   310|2.5296099185943604|    2|1.2648049592971802|
|   438|1.3243918418884277|    2|0.6621959209442139|
+------+------------------+-----+------------------+
only showing top 10 rows



# Next, we want to investigate whether there is a relationship between the number of reviews of a movie and the absolute error of the movie.

# Calculate, for each Movie, the sum of absolute prediction error, and the total number of reviews. Like calculating errors of users, we will then combine thse two DataFrames using join.

# Exercise 3 (15 points)
## Complete the code below to calculate the sum of absolute errors for each movie

In [32]:
Movie_Err_Sum = Abs_Err_DF.groupBy("MovieID").sum("AbsoluteErr")

In [33]:
Movie_Err_Sum.show(3)

+-------+-----------------+
|MovieID| sum(AbsoluteErr)|
+-------+-----------------+
| 160563|1.008368730545044|
|   1645|4.198756694793701|
|    471|7.425626993179321|
+-------+-----------------+
only showing top 3 rows



In [34]:
Movie_review_total= Abs_Err_DF.groupBy("MovieID").count()

In [35]:
Movie_review_total.show(3)

+-------+-----+
|MovieID|count|
+-------+-----+
| 160563|    1|
|   1645|   11|
|    471|   14|
+-------+-----+
only showing top 3 rows



In [36]:
Joined_Movie_Error_DF = Movie_Err_Sum.join(Movie_review_total, "MovieID", "inner")

In [37]:
Joined_Movie_Error_DF.show(5)

+-------+------------------+-----+
|MovieID|  sum(AbsoluteErr)|count|
+-------+------------------+-----+
| 160563| 1.008368730545044|    1|
|   1645| 4.198756694793701|   11|
|    471| 7.425626993179321|   14|
|  44022|1.5534019470214844|    4|
|   3175|11.340189456939697|   19|
+-------+------------------+-----+
only showing top 5 rows



# Compute the average absolute prediction errors for each movie

# Exercise 4 (15 points)
## Complete the code below to calculate the average absolute prediction error for each movie.

In [38]:
Avg_Movie_Error_DF = Joined_Movie_Error_DF.withColumn("AvgErr", col("sum(AbsoluteErr)")/col("count"))

In [39]:
Avg_Movie_Error_DF.show(3)

+-------+-----------------+-----+-------------------+
|MovieID| sum(AbsoluteErr)|count|             AvgErr|
+-------+-----------------+-----+-------------------+
| 160563|1.008368730545044|    1|  1.008368730545044|
|   1645|4.198756694793701|   11|0.38170515407215466|
|    471|7.425626993179321|   14| 0.5304019280842373|
+-------+-----------------+-----+-------------------+
only showing top 3 rows



In [40]:
sorted_Avg_Movie_Error_DF = Avg_Movie_Error_DF.orderBy("AvgErr", ascending=False)

In [41]:
sorted_Avg_Movie_Error_DF.show(10)

+-------+------------------+-----+------------------+
|MovieID|  sum(AbsoluteErr)|count|            AvgErr|
+-------+------------------+-----+------------------+
|   2824| 4.449501991271973|    1| 4.449501991271973|
|   1180| 4.063180923461914|    1| 4.063180923461914|
|     99| 7.601964950561523|    2|3.8009824752807617|
|  87522|3.8005995750427246|    1|3.8005995750427246|
|   4520|3.5214006900787354|    1|3.5214006900787354|
|   2902|3.5034680366516113|    1|3.5034680366516113|
|   3036| 6.955329895019531|    2|3.4776649475097656|
|  71033|3.3670318126678467|    1|3.3670318126678467|
| 117895| 3.365041732788086|    1| 3.365041732788086|
|   4486| 3.256599187850952|    1| 3.256599187850952|
+-------+------------------+-----+------------------+
only showing top 10 rows



In [42]:
Avg_Movie_Err_sorted_by_count_DF = Avg_Movie_Error_DF.orderBy("count", ascending=True)

In [43]:
Avg_Movie_Err_sorted_by_count_DF.show(10)

+-------+-------------------+-----+-------------------+
|MovieID|   sum(AbsoluteErr)|count|             AvgErr|
+-------+-------------------+-----+-------------------+
|    833|0.13506841659545898|    1|0.13506841659545898|
|   4519| 0.3032808303833008|    1| 0.3032808303833008|
|   3475| 0.4906930923461914|    1| 0.4906930923461914|
|   5803| 0.6063841581344604|    1| 0.6063841581344604|
|  36525| 1.7253344058990479|    1| 1.7253344058990479|
|   6466| 1.4032385349273682|    1| 1.4032385349273682|
|   2659| 0.6786079406738281|    1| 0.6786079406738281|
|   3794| 0.2716844081878662|    1| 0.2716844081878662|
| 160563|  1.008368730545044|    1|  1.008368730545044|
|   2122|0.13342857360839844|    1|0.13342857360839844|
+-------+-------------------+-----+-------------------+
only showing top 10 rows



In [44]:
from pyspark.sql.functions import avg

In [45]:
Avg_Movie_Error_DF.select( avg(col("AvgErr")) ).show()

+------------------+
|       avg(AvgErr)|
+------------------+
|0.7954067992405337|
+------------------+



# We want to calculate average prediction error for movies with k reviews (k=1, 2, 3, ..) to see whether there is a relationship between the number of reviews a movie received and its prediction error.

In [46]:
Avg_Movie_Error_DF2= Avg_Movie_Error_DF.withColumnRenamed("count", "ReviewCount")

In [47]:
Avg_Movie_Error_DF2.show(4)

+-------+------------------+-----------+-------------------+
|MovieID|  sum(AbsoluteErr)|ReviewCount|             AvgErr|
+-------+------------------+-----------+-------------------+
| 160563| 1.008368730545044|          1|  1.008368730545044|
|   1645| 4.198756694793701|         11|0.38170515407215466|
|    471| 7.425626993179321|         14| 0.5304019280842373|
|  44022|1.5534019470214844|          4| 0.3883504867553711|
+-------+------------------+-----------+-------------------+
only showing top 4 rows



# Exercise 5 (15 points)
## Complete the code below to calculate the sum of average prediction error for each k value (i.e., the column ``ReviewCount``).

In [48]:
Movie_k_review_AvgErr_Sum = Avg_Movie_Error_DF2.groupBy("ReviewCount").sum("AvgErr")

In [49]:
Movie_k_review_AvgErr_Sum.show(4)

+-----------+-----------------+
|ReviewCount|      sum(AvgErr)|
+-----------+-----------------+
|         26|4.270080456366906|
|         29|2.527662047024431|
|         65|0.531666766680204|
|         19|6.150488307601527|
+-----------+-----------------+
only showing top 4 rows



In [50]:
Movie_k_review_movie_count= Avg_Movie_Error_DF2.groupBy("ReviewCount").count()

In [51]:
Movie_k_review_movie_count.show(4)

+-----------+-----+
|ReviewCount|count|
+-----------+-----+
|         26|    7|
|         29|    3|
|         65|    1|
|         19|   10|
+-----------+-----+
only showing top 4 rows



In [52]:
Movie_k_review_joined_DF = Movie_k_review_AvgErr_Sum.join(Movie_k_review_movie_count, "ReviewCount", "inner")

In [53]:
Movie_k_review_joined_DF.show(10)

+-----------+------------------+-----+
|ReviewCount|       sum(AvgErr)|count|
+-----------+------------------+-----+
|         26| 4.270080456366906|    7|
|         29| 2.527662047024431|    3|
|         65| 0.531666766680204|    1|
|         19| 6.150488307601527|   10|
|         54|0.7651453327249598|    1|
|         22|  9.04955594106154|   13|
|          7| 86.71517193317409|  117|
|         34|2.3677817583084106|    4|
|         50|0.6330415821075439|    1|
|         57|0.7276243661579332|    1|
+-----------+------------------+-----+
only showing top 10 rows



# Exercise 6 (15 points)
## Complete the code below to calculate average prediction error for movies with k reviews, where k is the value in the column ``ReviewCount``.

In [54]:
Movie_k_review_AvgPredErr= Movie_k_review_joined_DF.withColumn("AvgPredErr_k_review_Movies", col("sum(AvgErr)")/col("count"))

In [55]:
Movie_k_review_AvgPredErr.show(10)

+-----------+------------------+-----+--------------------------+
|ReviewCount|       sum(AvgErr)|count|AvgPredErr_k_review_Movies|
+-----------+------------------+-----+--------------------------+
|         26| 4.270080456366906|    7|        0.6100114937667008|
|         29| 2.527662047024431|    3|        0.8425540156748103|
|         65| 0.531666766680204|    1|         0.531666766680204|
|         19| 6.150488307601527|   10|        0.6150488307601527|
|         54|0.7651453327249598|    1|        0.7651453327249598|
|         22|  9.04955594106154|   13|        0.6961196877739646|
|          7| 86.71517193317409|  117|        0.7411553156681546|
|         34|2.3677817583084106|    4|        0.5919454395771027|
|         50|0.6330415821075439|    1|        0.6330415821075439|
|         57|0.7276243661579332|    1|        0.7276243661579332|
+-----------+------------------+-----+--------------------------+
only showing top 10 rows



In [56]:
sorted_Movie_k_review_AvgPredErr = Movie_k_review_AvgPredErr.orderBy("ReviewCount", ascending = True)

In [57]:
sorted_Movie_k_review_AvgPredErr.show(100)

+-----------+------------------+-----+--------------------------+
|ReviewCount|       sum(AvgErr)|count|AvgPredErr_k_review_Movies|
+-----------+------------------+-----+--------------------------+
|          1|1286.6490778326988| 1470|        0.8752714815188427|
|          2| 596.7772898823023|  729|        0.8186245403049414|
|          3| 325.1669916411244|  430|        0.7562023061421498|
|          4|201.16979910433292|  275|        0.7315265421975743|
|          5|159.15083609819402|  215|        0.7402364469683442|
|          6|103.05308556556703|  140|        0.7360934683254788|
|          7| 86.71517193317409|  117|        0.7411553156681546|
|          8|  71.0319482833147|  103|        0.6896305658574243|
|          9| 60.24144793881311|   84|        0.7171600945096799|
|         10| 40.77164438962936|   59|        0.6910448201632095|
|         11| 39.24833058227193|   55|        0.7136060105867624|
|         12| 38.03928248087565|   53|        0.7177223109599179|
|         

# Exercise 6 (15 points)
## Describe the relationship between number of reviews a movie (in the testing data) received and the prediction error of the movie.

## Answer to Exercise 6:
- As the review count goes up, generally the average prediction error of the movie goes down. There is a little fluctuation but this is the case for most of these test data points.

# Finally, we want to investigate whether the prediction error varies significantly among different genres.

In [58]:
movie_schema = StructType([ StructField("MovieID", IntegerType(), False), \
                            StructField("MovieTitle", StringType(), True ), \
                            StructField("Genres", StringType(), True ), \
                           ])

In [60]:
movies_DF = ss.read.csv("/storage/home/ajv5723/work/Lab5/movies_samples.csv", schema=movie_schema, header=True, inferSchema=False)
# In the cluster mode, we need to change the input path as well as the header parameter: `header=False` because the large movie file does not have header.

In [61]:
movies_DF.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- MovieTitle: string (nullable = true)
 |-- Genres: string (nullable = true)



In [62]:
movies_DF.show(10)

+-------+--------------------+--------------------+
|MovieID|          MovieTitle|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
+-------+--------------------+--------------------+
only showing top 10 rows



In [63]:
from pyspark.sql.functions import split

In [64]:
movies_GArray_DF = movies_DF.withColumn("GenresArray", split(col("Genres"), '\|') )

In [65]:
movies_GArray_DF.show(3)

+-------+--------------------+--------------------+--------------------+
|MovieID|          MovieTitle|              Genres|         GenresArray|
+-------+--------------------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|[Adventure, Anima...|
|      2|      Jumanji (1995)|Adventure|Childre...|[Adventure, Child...|
|      3|Grumpier Old Men ...|      Comedy|Romance|   [Comedy, Romance]|
+-------+--------------------+--------------------+--------------------+
only showing top 3 rows



# Join Prediction Errors of Movies in Testing Data with the Movie_GArray_DF to obtain its GenresArray

In [66]:
Avg_Movie_Error_DF2.show(4)

+-------+------------------+-----------+-------------------+
|MovieID|  sum(AbsoluteErr)|ReviewCount|             AvgErr|
+-------+------------------+-----------+-------------------+
| 160563| 1.008368730545044|          1|  1.008368730545044|
|   1645| 4.198756694793701|         11|0.38170515407215466|
|    471| 7.425626993179321|         14| 0.5304019280842373|
|  44022|1.5534019470214844|          4| 0.3883504867553711|
+-------+------------------+-----------+-------------------+
only showing top 4 rows



In [67]:
joined_Movie_Genre_AvgErr_DF = Avg_Movie_Error_DF2.join(movies_GArray_DF, "MovieID", "inner")

In [68]:
joined_Movie_Genre_AvgErr_DF.show(4)

+-------+------------------+-----------+-------------------+--------------------+--------------------+--------------------+
|MovieID|  sum(AbsoluteErr)|ReviewCount|             AvgErr|          MovieTitle|              Genres|         GenresArray|
+-------+------------------+-----------+-------------------+--------------------+--------------------+--------------------+
| 160563| 1.008368730545044|          1|  1.008368730545044|The Legend of Tar...|    Action|Adventure| [Action, Adventure]|
|   1645| 4.198756694793701|         11|0.38170515407215466|The Devil's Advoc...|Drama|Mystery|Thr...|[Drama, Mystery, ...|
|    471| 7.425626993179321|         14| 0.5304019280842373|Hudsucker Proxy, ...|              Comedy|            [Comedy]|
|  44022|1.5534019470214844|          4| 0.3883504867553711|Ice Age 2: The Me...|Adventure|Animati...|[Adventure, Anima...|
+-------+------------------+-----------+-------------------+--------------------+--------------------+--------------------+
only sho

# Obtain the list of all genres

In [69]:
genres_rdd = movies_DF.select("Genres").rdd

In [70]:
genres_rdd.take(3)

[Row(Genres='Adventure|Animation|Children|Comedy|Fantasy'),
 Row(Genres='Adventure|Children|Fantasy'),
 Row(Genres='Comedy|Romance')]

In [71]:
genres_flatten_rdd = genres_rdd.flatMap(lambda x: x["Genres"].split("|") )

In [72]:
genres_flatten_rdd.take(10)

['Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Fantasy',
 'Adventure',
 'Children',
 'Fantasy',
 'Comedy',
 'Romance']

In [73]:
genres_rdd = genres_flatten_rdd.distinct()

In [74]:
genres_list = genres_rdd.collect()

In [75]:
print(genres_list)

['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy', 'Romance', 'Drama', 'Action', 'Crime', 'Thriller', 'Horror', 'Mystery', 'Sci-Fi', 'Documentary', 'IMAX', 'War', 'Musical', 'Western', 'Film-Noir', '(no genres listed)']


# The following code demonstrates how to filter movies of a specific genre, and calculate the average prediction error across all movies of that genre.

In [76]:
from pyspark.sql.functions import array_contains

In [77]:
genre = genres_list[0]
genre_movie_DF = joined_Movie_Genre_AvgErr_DF.filter(array_contains(col("GenresArray"), genre) )

In [78]:
genre_movie_DF.show(3)

+-------+------------------+-----------+------------------+--------------------+--------------------+--------------------+
|MovieID|  sum(AbsoluteErr)|ReviewCount|            AvgErr|          MovieTitle|              Genres|         GenresArray|
+-------+------------------+-----------+------------------+--------------------+--------------------+--------------------+
| 160563| 1.008368730545044|          1| 1.008368730545044|The Legend of Tar...|    Action|Adventure| [Action, Adventure]|
|  44022|1.5534019470214844|          4|0.3883504867553711|Ice Age 2: The Me...|Adventure|Animati...|[Adventure, Anima...|
|   3175|11.340189456939697|         19|0.5968520766810367| Galaxy Quest (1999)|Adventure|Comedy|...|[Adventure, Comed...|
+-------+------------------+-----------+------------------+--------------------+--------------------+--------------------+
only showing top 3 rows



In [79]:
g_movie_avg_err = genre_movie_DF.select( avg(col("AvgErr")) ).rdd

In [80]:
g_movie_avg_err.take(1)

[Row(avg(AvgErr)=0.7822195362189941)]

In [81]:
Avg_Error_for_g_movies = g_movie_avg_err.take(1)[0]["avg(AvgErr)"]

In [82]:
print(Avg_Error_for_g_movies)

0.7822195362189941


# Calculate the average prediction error of movies in the testing data for each genre.
# Exercise 7 (15 points)
## Complete the code below for calculating average prediction error of movies in the testing data for each genre.

In [84]:
for genre in genres_list:
    # Filter for movies of the genre
    genre_movie_DF = joined_Movie_Genre_AvgErr_DF.filter(array_contains(col("GenresArray"), genre) )
    g_movie_avg_err = genre_movie_DF.select( avg(col("AvgErr")) ).rdd
    Avg_Error_for_g_movies = g_movie_avg_err.take(1)[0]["avg(AvgErr)"]
    print("Average Prediction Error of Movies in the Testing Data for Genre", genre, " = ", Avg_Error_for_g_movies)                              

Average Prediction Error of Movies in the Testing Data for Genre Adventure  =  0.7822195362189941
Average Prediction Error of Movies in the Testing Data for Genre Animation  =  0.7443807762910141
Average Prediction Error of Movies in the Testing Data for Genre Children  =  0.7876774305492731
Average Prediction Error of Movies in the Testing Data for Genre Comedy  =  0.8003410699777272
Average Prediction Error of Movies in the Testing Data for Genre Fantasy  =  0.8023302410049794
Average Prediction Error of Movies in the Testing Data for Genre Romance  =  0.7875969511585504
Average Prediction Error of Movies in the Testing Data for Genre Drama  =  0.7830962112288482
Average Prediction Error of Movies in the Testing Data for Genre Action  =  0.7768770350969592
Average Prediction Error of Movies in the Testing Data for Genre Crime  =  0.7627975604152715
Average Prediction Error of Movies in the Testing Data for Genre Thriller  =  0.7465758732092604
Average Prediction Error of Movies in th

# What is the average absolute prediction error across all movies?

In [86]:

g_movie_avg_err = joined_Movie_Genre_AvgErr_DF.select( avg(col("AvgErr")) ).rdd
Avg_Error_for_g_movies = g_movie_avg_err.take(1)[0]["avg(AvgErr)"]
print("Average absolute Prediction Error across all movies", " = ", Avg_Error_for_g_movies)                              

Average absolute Prediction Error across all movies  =  0.7954067992405337


# Exercise 8 (10 points)
## What are three genres, on the average, harder to predict ratings than movies in other genres?
## Answer to Exercise 8:

### The three genres that are harder to predict on average are Horror(0.8410), Comedy(0.8003), and Documentary(0.8675)

In [87]:
ss.stop()